In [ ]:
# Get BORI <--> KMG <--> Dutt Title mapping

cross_edition__title_map = []

def get_title_numbers(line):
    ls = line.split('</div>')
    
    a = ls[0]
    ali = a.rfind('>')
    bori_ch_id = a[ali+1:]
    
    b = ls[1]
    bli = b.rfind('>')
    kmg_ch_id = b[bli+1:]
    
    c = ls[2]
    cli = c.rfind('>')
    dutt_ch_id = c[cli+1:]
    
    return([bori_ch_id, kmg_ch_id, dutt_ch_id])
    
with open('svat_html.txt', 'r') as file:
        for idx, line in enumerate(file):
            if line.startswith('<tr id='):
                cross_edition__title_map.append(get_title_numbers(line))
                # break
                
len(cross_edition__title_map)
cross_edition__title_map_df = pd.DataFrame(cross_edition__title_map, columns=['bori_chapter_ids', 'kmg_chapter_ids', 'dutt_chapter_names'])
cross_edition__title_map_df.to_csv('bori_kmg_dutt_chapter_map.csv', index=False)

In [ ]:
# extract google_trans from old jsons

old_data_path = "formatted_json_data_old"
book_jsons = os.listdir(old_data_path)

all_gtrans_data = []
for bk in book_jsons:
    bk_path = os.path.join(old_data_path, bk)
    
    with open(bk_path, 'r', encoding='utf-8') as f:
        bk_obj = json.load(f)
        book_id = str(bk_obj['book_number']).zfill(2)
        for c in bk_obj['chapters']:
            chapter_id = c['chapter_name'][-3:]
            for v in c['verses']:
                verse_id = v['verse_number']
                verse_uid = book_id+chapter_id+verse_id
                v_sans = '। '.join(v['verse_data']) + '।'
                v_gtrans = v['verse_translation']['google_trans']
                
                all_gtrans_data.append([verse_uid, v_sans, v_gtrans])
                

# all_bori_gtrans_data_df = pd.DataFrame(all_gtrans_data, columns=['bori_id', 'sans', 'gtrans'])
# all_bori_gtrans_data_df.to_csv('all_bori_gtrans_data.csv', index=False)

all_gtrans_data_no_headings = []
heading_row = []
for row in all_gtrans_data:
    v_id = row[0]
    
    if v_id.endswith('h'):
        heading_row = row.copy()
    else:
        if heading_row:
            extended_sans = heading_row[1] + ' ' + row[1]
            extended_gtrans = heading_row[2] + ': ' + row[2]
            new_row = [v_id, extended_sans, extended_gtrans]
            all_gtrans_data_no_headings.append(row)
            heading_row = []
        else:
            all_gtrans_data_no_headings.append(row)
            
all_bori_gtrans_nh_data_df = pd.DataFrame(all_gtrans_data_no_headings, columns=['bori_id', 'sans', 'gtrans'])
all_bori_gtrans_nh_data_df.to_csv('all_bori_gtrans_data.csv', index=False)